# FirstAI / Orbit — independent Colab T4 training
Build sourced datasets, train FirstGPT from random initialization, instruction-tune Orbit, then download its checkpoint. No Drive mount or manual dataset uploads. Colab storage is temporary.

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO=Path('/content/FirstAI')
if REPO.exists(): subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','https://github.com/llakshyaa-rgb/FirstAI.git',str(REPO)],check=True)
os.chdir(REPO); sys.path.insert(0,str(REPO))
import torch
print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda)
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime → Change runtime type → T4 GPU, reconnect, and rerun.')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
subprocess.run([sys.executable,'-m','pip','install','-q','datasets>=3.0'],check=True)


## 1. Download and prepare a larger sourced dataset mix
Defaults: up to 30 million WikiText characters, up to 10,000 training rows per capability dataset and up to 10,000 examples per source after balancing. These caps are configurable; watch Colab disk/RAM. The live-data lab adds current public observations and derived real-life scenarios.


In [ ]:
subprocess.run([sys.executable,'build_all_data.py','--country','IND','--city','New Delhi','--max-per-dataset','10000','--max-base-chars','30000000','--per-source-cap','10000'],check=True)
import json
for f in ['data/capabilities/manifest.json','data/instructions_balanced.manifest.json']:
 p=Path(f)
 if p.exists(): print('\n'+f, json.dumps(json.loads(p.read_text()),indent=2)[:12000])


## 2. Train FirstGPT from scratch — resumable
Default profile: 10,000 additional optimizer steps, six layers, width 256, 1,024-token context, mixed precision and periodic checkpoints. Training resumes from the last checkpoint if runtime files persist. This is still a small experimental model, not a modern foundation model.


In [ ]:
base_args=['--data','data/input.txt','--out','checkpoints/firstgpt-real.pt','--steps',os.environ.get('FIRSTAI_BASE_STEPS','10000'),'--eval-interval','200','--checkpoint-interval','250','--eval-batches','12','--batch-size','4','--grad-accum-steps','4','--block-size','1024','--layers','6','--heads','8','--width','256','--vocab-size','512','--amp']
if Path('checkpoints/firstgpt-real.pt').exists() and os.environ.get('FIRSTAI_RESTART_BASE','0')!='1': base_args.append('--resume')
subprocess.run([sys.executable,'train.py',*base_args],check=True)


## 3. Instruction-tune on practical tasks — resumable
The latest SFT checkpoint can resume after interruption when the dataset hash is unchanged. The best validation checkpoint is saved separately.


In [ ]:
sft_args=['--checkpoint','checkpoints/firstgpt-real.best.pt','--data','data/instructions_balanced.jsonl','--out','checkpoints/orbit-instruct.pt','--steps',os.environ.get('FIRSTAI_SFT_STEPS','5000'),'--batch-size','4','--eval-interval','100']
if Path('checkpoints/orbit-instruct.pt').exists() and os.environ.get('FIRSTAI_RESTART_SFT','0')!='1': sft_args.append('--resume')
subprocess.run([sys.executable,'finetune_instructions.py',*sft_args],check=True)
print('Best SFT checkpoint:',Path('checkpoints/orbit-instruct.best.pt').stat().st_size,'bytes')


## 4. Download both checkpoints
Download before disconnecting; Colab's temporary disk is erased when the runtime ends.

In [ ]:
from google.colab import files
files.download('checkpoints/firstgpt-real.best.pt')
files.download('checkpoints/orbit-instruct.best.pt')
